# Long-Term Memory with LangMem

**Thread memory** (checkpointers) dies when you start a new chat. **Long-term memory** survives across threads, sessions, and days — keyed by **user**, not `thread_id`.

**LangMem** is LangChain's SDK for agent long-term memory. It sits on top of LangGraph's `BaseStore` and provides:

| Capability | LangMem construct |
|------------|-------------------|
| Agent writes memory in conversation | `create_manage_memory_tool` (hot path) |
| Agent searches past facts | `create_search_memory_tool` |
| Background extraction after chat | `create_memory_manager` / `create_memory_store_manager` |
| Structured memories | Pydantic `schema` on tools and managers |

```
  thread_id=chat-42 (ephemeral)          user_id=alice (durable)
         │                                       │
         ▼                                       ▼
   Checkpointer                         LangMem + BaseStore
   ─ messages this chat                 ─ namespace: ("shopco", "alice")
   ─ order_id scratch                   ─ semantic: prefers email contact
                                        ─ episodic: ORD-1001 return 2026-07
                                        ─ procedural: concise answers
```

This notebook builds **ShopCo Support** with LangMem patterns:

1. Hot-path `manage_memory` / `search_memory` tools
2. Background memory extraction (offline rules + optional live LLM)
3. Cross-thread recall with namespace isolation
4. LangGraph agent with `store` + `checkpointer`

Offline by default; optional live LangMem at the end.

In [ ]:
"""
ShopCo Support — LangMem long-term memory lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Annotated, Any, Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
LANGMEM_NAMESPACE = ("shopco", "{langgraph_user_id}")


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
    "billing": "Charges appear as SHOPCO* on statements. [pol-billing-03]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "email": "alice@shopco.com"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "email": "bob@shopco.com"},
}

print("ShopCo LangMem environment ready.")

---

## 1. LangMem vs Raw BaseStore

Raw `BaseStore` usage (`put` / `get` / `search`) works but leaves memory management to you. LangMem adds:

- **Tools** the agent can call to create/update/delete memories mid-conversation
- **Semantic search** over stored memories (with vector index on `InMemoryStore`)
- **Background managers** that extract facts from full conversation transcripts
- **Consolidation** — update or supersede outdated memories instead of duplicating

LangMem does not replace checkpointers. Use **both**: checkpointer for the active thread; LangMem + store for durable user knowledge.

---

## 2. Memory Types LangMem Supports

| Type | What to store | ShopCo example |
|------|---------------|----------------|
| **Semantic** | Stable facts and preferences | "Prefers email over phone" |
| **Episodic** | Specific past events | "Returned ORD-8842 in June" |
| **Procedural** | How the agent should behave | "Give concise answers" |

LangMem stores these as items in `BaseStore` namespaces. Type is metadata you attach via schema or conventions.

In [ ]:
class ShopCoMemory(BaseModel):
    """Structured memory schema — mirrors LangMem's optional Pydantic schema."""
    content: str
    memory_type: Literal["semantic", "episodic", "procedural"] = "semantic"
    source: str = "conversation"
    confidence: float = Field(ge=0.0, le=1.0, default=0.9)


example = ShopCoMemory(
    content="Customer prefers email contact",
    memory_type="semantic",
)
print(example.model_dump())

---

## 3. Namespaces — User-Scoped, Not Thread-Scoped

LangMem namespaces use runtime placeholders like `{langgraph_user_id}`:

```python
namespace=("shopco", "{langgraph_user_id}")
config = {"configurable": {"langgraph_user_id": "alice"}}
# Resolves to ("shopco", "alice")
```

Never key long-term memory by `thread_id` — users open many threads over time.

In [ ]:
def resolve_namespace(template: tuple[str, ...], config: RunnableConfig) -> tuple[str, ...]:
    user_id = config.get("configurable", {}).get("langgraph_user_id", "anonymous")
    return tuple(
        part.format(langgraph_user_id=user_id) if "{" in part else part
        for part in template
    )


LTM_STORE = InMemoryStore()

cfg_alice = {"configurable": {"langgraph_user_id": "alice"}}
ns_alice = resolve_namespace(LANGMEM_NAMESPACE, cfg_alice)
LTM_STORE.put(ns_alice, "mem-pref-01", ShopCoMemory(
    content="Prefers concise answers", memory_type="procedural",
).model_dump())

print("Alice namespace:", ns_alice)
print("Stored:", LTM_STORE.get(ns_alice, "mem-pref-01").value)

---

## 4. Hot Path vs Background Writes

| Pattern | When | Latency |
|---------|------|---------|
| **Hot path** | Agent calls `manage_memory` during chat | Adds to turn time |
| **Background** | `create_memory_manager` runs after session ends | Zero user-facing latency |

Use hot path for explicit "remember this" requests. Use background for extracting implicit facts from long transcripts.

---

## 5. Offline LangMem Tools — `manage_memory` and `search_memory`

These mirror the LangMem tool signatures so the notebook runs without API keys. When `USE_LIVE_LLM=True`, you can swap in `create_manage_memory_tool` / `create_search_memory_tool` from the `langmem` package.

In [ ]:
MemoryAction = Literal["create", "update", "delete"]


def _tokenize(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2}


def manage_memory(
    *,
    store: BaseStore,
    namespace: tuple[str, ...],
    action: MemoryAction = "create",
    content: str | None = None,
    memory_id: str | None = None,
    memory_type: Literal["semantic", "episodic", "procedural"] = "semantic",
) -> str:
    """LangMem-compatible manage_memory (offline implementation)."""
    if action == "create":
        mid = memory_id or f"mem-{uuid.uuid4().hex[:8]}"
        record = ShopCoMemory(content=content or "", memory_type=memory_type).model_dump()
        record["created_at"] = utc_now()
        store.put(namespace, mid, record)
        return f"created memory {mid}"
    if action == "update":
        if not memory_id:
            return "error: id required for update"
        existing = store.get(namespace, memory_id)
        if not existing:
            return f"error: memory {memory_id} not found"
        record = dict(existing.value)
        if content:
            record["content"] = content
        record["updated_at"] = utc_now()
        store.put(namespace, memory_id, record)
        return f"updated memory {memory_id}"
    if action == "delete":
        if not memory_id:
            return "error: id required for delete"
        store.delete(namespace, memory_id)
        return f"deleted memory {memory_id}"
    return "error: unknown action"


def search_memory(
    *,
    store: BaseStore,
    namespace: tuple[str, ...],
    query: str,
    limit: int = 5,
) -> list[dict[str, Any]]:
    """LangMem-compatible search_memory (offline keyword scoring)."""
    items = store.search(namespace)
    q_tokens = _tokenize(query)
    scored: list[tuple[float, Any]] = []
    for item in items:
        body = str(item.value.get("content", item.value))
        overlap = len(q_tokens & _tokenize(body))
        if overlap > 0:
            scored.append((overlap / max(len(q_tokens), 1), item))
    scored.sort(key=lambda x: x[0], reverse=True)
    results = []
    for score, item in scored[:limit]:
        results.append({"id": item.key, "score": round(score, 3), "value": item.value})
    return results


test_ns = resolve_namespace(LANGMEM_NAMESPACE, cfg_alice)
print(manage_memory(store=LTM_STORE, namespace=test_ns, content="Contact via email only", memory_type="semantic"))
print(search_memory(store=LTM_STORE, namespace=test_ns, query="email contact"))

---

## 6. Background Memory Extractor (Offline Rules)

Mirrors `create_memory_manager` — processes a conversation transcript and returns memory operations without blocking the user.

In [ ]:
@dataclass
class ExtractedMemory:
    action: MemoryAction
    content: str
    memory_type: Literal["semantic", "episodic", "procedural"]
    memory_id: str | None = None
    reason: str = ""


PREFERENCE_PATTERNS = [
    (r"prefer\s+(\w+(?:\s+\w+)?)", "semantic", "preference stated"),
    (r"remember\s+(?:that\s+)?(.+)", "semantic", "explicit remember request"),
    (r"(?:be\s+)?concise|short answers", "procedural", "response style"),
    (r"ORD-[0-9]{4}.*return", "episodic", "order event"),
]


def extract_memories_offline(messages: list[dict[str, str]], existing: list[dict[str, Any]] | None = None) -> list[ExtractedMemory]:
    """Rule-based stand-in for create_memory_manager (offline)."""
    existing = existing or []
    extracted: list[ExtractedMemory] = []
    for msg in messages:
        if msg.get("role") != "user":
            continue
        text = msg["content"]
        for pattern, mtype, reason in PREFERENCE_PATTERNS:
            m = re.search(pattern, text, re.I)
            if not m:
                continue
            content = m.group(0) if mtype == "episodic" else m.group(1) if m.lastindex else text
            # Check for contradiction / update
            for ex in existing:
                old_content = str(ex.get("value", {}).get("content", ""))
                if mtype == "semantic" and "prefer" in old_content.lower() and "prefer" in text.lower():
                    extracted.append(ExtractedMemory(
                        action="update", content=text, memory_type=mtype,
                        memory_id=ex.get("id"), reason="preference updated",
                    ))
                    break
            else:
                extracted.append(ExtractedMemory(action="create", content=content.strip(), memory_type=mtype, reason=reason))
    return extracted


def apply_extracted_memories(store: BaseStore, namespace: tuple[str, ...], items: list[ExtractedMemory]) -> list[str]:
    trace: list[str] = []
    for item in items:
        result = manage_memory(
            store=store, namespace=namespace, action=item.action,
            content=item.content, memory_id=item.memory_id, memory_type=item.memory_type,
        )
        trace.append(f"{item.reason}:{result}")
    return trace


conv = [{"role": "user", "content": "Please remember I prefer email over phone for contact"}]
items = extract_memories_offline(conv)
print("Extracted:", [(i.action, i.content, i.memory_type) for i in items])

---

## 7. Graph State — Thread Scratch + Loaded Memories

In [ ]:
class LangMemState(TypedDict):
    messages: Annotated[list, add_messages]
    order_id: str
    customer_name: str
    turn_count: int
    memory_context: str
    memory_trace: list[str]


def extract_thread_entities(state: LangMemState) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last:
        return {"turn_count": state.get("turn_count", 0)}
    text = str(last.content)
    updates: dict[str, Any] = {"turn_count": state.get("turn_count", 0) + 1}
    m = re.search(r"ORD-[0-9]{4}", text.upper())
    if m:
        updates["order_id"] = m.group(0)
    name_m = re.search(r"my name is ([A-Za-z]+)", text, re.I)
    if name_m:
        updates["customer_name"] = name_m.group(1)
    return updates

---

## 8. Load Memories — Inject Long-Term Context Each Turn

In [ ]:
def load_memories_node(state: LangMemState, config: RunnableConfig, *, store: BaseStore) -> dict[str, Any]:
    ns = resolve_namespace(LANGMEM_NAMESPACE, config)
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    query = str(last.content) if last else "preferences"
    hits = search_memory(store=store, namespace=ns, query=query, limit=5)
    if not hits:
        all_items = store.search(ns)
        hits = [{"id": i.key, "score": 0.0, "value": i.value} for i in all_items[:5]]
    lines = [f"[{h['id']}] ({h['value'].get('memory_type', '?')}) {h['value'].get('content', h['value'])}" for h in hits]
    ctx = "\n".join(lines) if lines else "(no long-term memories)"
    return {"memory_context": ctx, "memory_trace": state.get("memory_trace", []) + [f"load:{len(hits)} hits"]}

---

## 9. Hot-Path Memory Node — Agent Decides to Write

In [ ]:
def hot_path_memory_node(state: LangMemState, config: RunnableConfig, *, store: BaseStore) -> dict[str, Any]:
    """Detect explicit remember/prefer requests and write via manage_memory (hot path)."""
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last:
        return {}
    text = str(last.content)
    ns = resolve_namespace(LANGMEM_NAMESPACE, config)
    trace = list(state.get("memory_trace", []))

    if "remember" in text.lower() or "prefer" in text.lower():
        mtype: Literal["semantic", "episodic", "procedural"] = "procedural" if "concise" in text.lower() else "semantic"
        result = manage_memory(store=store, namespace=ns, content=text, memory_type=mtype)
        trace.append(f"hot_path:{result}")
        return {"memory_trace": trace, "messages": [ToolMessage(content=result, tool_call_id="manage_memory")]}
    return {}


def respond_node(state: LangMemState, config: RunnableConfig) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last.content).lower() if last else ""
    name = state.get("customer_name") or "there"
    order_id = state.get("order_id", "")
    mem = state.get("memory_context", "")

    if "return" in q:
        body = POLICY_SNIPPETS["returns"]
    elif "shipping" in q:
        body = POLICY_SNIPPETS["shipping"]
    elif "status" in q or ("it" in q and order_id):
        row = ORDERS_DB.get(order_id, {})
        body = f"Order {order_id or '?'} status: {row.get('status', 'unknown')}."
    elif "remember" in q or "prefer" in q:
        body = "Got it — I've saved that to your long-term memory for future chats."
    elif "what do you know" in q or "my preferences" in q:
        body = f"Here's what I have on file:\n{mem}"
    else:
        body = "How can I help with your ShopCo order or policy?"

    if mem and mem != "(no long-term memories)" and "what do you know" not in q:
        if "procedural" in mem and "concise" in mem.lower():
            body = body.split(".")[0] + "."  # honor procedural memory: be concise

    return {"messages": [AIMessage(content=f"Hi {name}, {body}")]}

---

## 10. Background Memory Node — Post-Turn Extraction

In [ ]:
def background_memory_node(state: LangMemState, config: RunnableConfig, *, store: BaseStore) -> dict[str, Any]:
    """Run after respond — extract implicit facts (background path)."""
    ns = resolve_namespace(LANGMEM_NAMESPACE, config)
    existing_hits = search_memory(store=store, namespace=ns, query="preference", limit=10)
    messages = []
    for m in state["messages"][-4:]:
        if isinstance(m, HumanMessage):
            messages.append({"role": "user", "content": str(m.content)})
        elif isinstance(m, AIMessage):
            messages.append({"role": "assistant", "content": str(m.content)})
    extracted = extract_memories_offline(messages, existing_hits)
    # Skip if hot path already handled explicit remember
    last_human = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if last_human and "remember" in str(last_human.content).lower():
        return {}
    trace = apply_extracted_memories(store, ns, extracted)
    if trace:
        return {"memory_trace": state.get("memory_trace", []) + [f"background:{t}" for t in trace]}
    return {}

---

## 11. Compile LangGraph — Checkpointer + Store

In [ ]:
def build_langmem_graph(store: BaseStore, checkpointer: MemorySaver | None = None):
    builder = StateGraph(LangMemState)
    builder.add_node("load_memories", load_memories_node)
    builder.add_node("extract", extract_thread_entities)
    builder.add_node("hot_path", hot_path_memory_node)
    builder.add_node("respond", respond_node)
    builder.add_node("background", background_memory_node)

    builder.add_edge(START, "load_memories")
    builder.add_edge("load_memories", "extract")
    builder.add_edge("extract", "hot_path")
    builder.add_edge("hot_path", "respond")
    builder.add_edge("respond", "background")
    builder.add_edge("background", END)

    return builder.compile(checkpointer=checkpointer, store=store)


SHARED_STORE = InMemoryStore()
CHECKPOINTER = MemorySaver()
GRAPH = build_langmem_graph(store=SHARED_STORE, checkpointer=CHECKPOINTER)
print("LangMem graph compiled with MemorySaver + InMemoryStore")

---

## 12. Invoke Helper — `thread_id` + `langgraph_user_id`

In [ ]:
def send_message(
    graph,
    user_id: str,
    thread_id: str,
    text: str,
    state: LangMemState | None = None,
) -> LangMemState:
    config: RunnableConfig = {
        "configurable": {"thread_id": thread_id, "langgraph_user_id": user_id},
    }
    payload: dict[str, Any] = {"messages": [HumanMessage(content=text)]}
    if state:
        for key in ("order_id", "customer_name", "turn_count", "memory_trace"):
            if state.get(key):
                payload[key] = state[key]
    return graph.invoke(payload, config=config)


def last_ai(state: LangMemState) -> str:
    for m in reversed(state["messages"]):
        if isinstance(m, AIMessage):
            return str(m.content)
    return ""

---

## 13. Demo — Hot Path Write in Thread 1

In [ ]:
s1 = send_message(GRAPH, "alice", "chat-monday", "My name is Alice. Remember I prefer email contact.")
print(last_ai(s1))
print("Trace:", s1.get("memory_trace", []))

ns = resolve_namespace(LANGMEM_NAMESPACE, {"configurable": {"langgraph_user_id": "alice"}})
alice_memories = SHARED_STORE.search(ns)
print("Alice memories:", [(m.key, m.value.get("content", "")[:50]) for m in alice_memories])

---

## 14. Cross-Thread Recall — New Chat, Same User

In [ ]:
# New thread_id — thread checkpoint is empty, but long-term memory persists
s2 = send_message(GRAPH, "alice", "chat-friday", "What do you know about my preferences?")
print(last_ai(s2))
print("\nMemory context loaded:")
print(s2.get("memory_context", "")[:300])

---

## 15. Namespace Isolation — Bob Cannot See Alice's Memories

In [ ]:
s_bob = send_message(GRAPH, "bob", "bob-chat-1", "What are my contact preferences?")
print("Bob:", last_ai(s_bob))
print("Bob memory context:", s_bob.get("memory_context"))

ns_bob = resolve_namespace(LANGMEM_NAMESPACE, {"configurable": {"langgraph_user_id": "bob"}})
print("Bob store items:", [m.key for m in SHARED_STORE.search(ns_bob)])

---

## 16. Memory Update — Preference Changes

In [ ]:
s3 = send_message(GRAPH, "alice", "chat-monday", "Actually I now prefer phone contact over email.", state=s1)
print(last_ai(s3))

hits = search_memory(
    store=SHARED_STORE,
    namespace=resolve_namespace(LANGMEM_NAMESPACE, {"configurable": {"langgraph_user_id": "alice"}}),
    query="contact phone email",
)
print("\nUpdated memories matching contact:")
for h in hits:
    print(f"  {h['id']}: {h['value'].get('content', '')[:60]}")

---

## 17. Procedural Memory — Response Style

In [ ]:
s4 = send_message(GRAPH, "alice", "chat-friday", "Please be concise from now on.", state=s2)
s5 = send_message(GRAPH, "alice", "chat-friday", "What is the return policy?", state=s4)
print("Concise response:", last_ai(s5))
print("Trace:", s5.get("memory_trace", [])[-3:])

---

## 18. Eval Harness

In [ ]:
def count_memories(store: BaseStore, user_id: str) -> int:
    ns = resolve_namespace(LANGMEM_NAMESPACE, {"configurable": {"langgraph_user_id": user_id}})
    return len(store.search(ns))


EVAL_CHECKS = [
    ("alice has memories", lambda: count_memories(SHARED_STORE, "alice") >= 1),
    ("bob isolated", lambda: count_memories(SHARED_STORE, "bob") == 0),
    ("cross-thread recall", lambda: "email" in s2.get("memory_context", "").lower() or "phone" in s2.get("memory_context", "").lower()),
    ("procedural stored", lambda: any("procedural" in str(m.value) for m in SHARED_STORE.search(
        resolve_namespace(LANGMEM_NAMESPACE, {"configurable": {"langgraph_user_id": "alice"}})
    ))),
]

passed = sum(int(fn()) for _, fn in EVAL_CHECKS)
for label, fn in EVAL_CHECKS:
    print(f"{'✓' if fn() else '✗'} {label}")
print(f"\nEval: {passed}/{len(EVAL_CHECKS)}")

---

## 19. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Key LTM by `thread_id` | Memory lost on new chat | Use `langgraph_user_id` namespace |
| Only hot path, no background | Miss implicit facts | Add `create_memory_manager` after sessions |
| Duplicate memories | Contradictory facts pile up | Enable updates; consolidate |
| No search before answer | Agent ignores stored prefs | `load_memories` node each turn |
| Store without checkpointer | Thread state lost on crash | Compile with **both** |
| Unbounded memory growth | Stale facts forever | Governance: decay, audit trails |

---

## 20. Production Checklist

- [ ] `langmem` installed; `InMemoryStore` replaced with `PostgresStore` + embedding index
- [ ] Namespace template: `("app", "{langgraph_user_id}")` — never `thread_id`
- [ ] Hot path: `create_manage_memory_tool` + `create_search_memory_tool` on agent
- [ ] Background: `create_memory_store_manager` on session end
- [ ] Structured schemas (`ShopCoMemory` or domain Pydantic models)
- [ ] Graph compiled with `checkpointer` + `store`
- [ ] Config passes both `thread_id` and `langgraph_user_id`
- [ ] Memory governance: conflicts, decay, audit trail

---

## 21. Optional Live LangMem

Set `USE_LIVE_LLM = True` to use the real LangMem SDK with OpenAI embeddings on `InMemoryStore`.

In [ ]:
def build_live_langmem_agent():
    """Requires: pip install langmem langchain-openai"""
    if not USE_LIVE_LLM:
        print("Set USE_LIVE_LLM = True to run live LangMem.")
        return None

    from langchain_openai import ChatOpenAI
    from langgraph.prebuilt import create_react_agent
    from langmem import create_manage_memory_tool, create_search_memory_tool

    store = InMemoryStore(index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    })
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    tools = [
        create_manage_memory_tool(namespace=LANGMEM_NAMESPACE, schema=ShopCoMemory),
        create_search_memory_tool(namespace=LANGMEM_NAMESPACE),
    ]
    agent = create_react_agent(llm, tools=tools, store=store)
    config = {"configurable": {"langgraph_user_id": "alice-live"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Remember I prefer email contact for ShopCo support."}]},
        config=config,
    )
    print(result["messages"][-1].content)
    return agent


build_live_langmem_agent()

---

## 22. Quiz

1. What is the difference between LangMem namespaces and checkpointer `thread_id`?
2. When should you use hot-path `manage_memory` vs background `create_memory_manager`?
3. How does LangMem handle a user changing their stated preference?
4. What three memory types did we model with `ShopCoMemory`?
5. Why compile the graph with both `checkpointer` and `store`?

---

## 23. Summary

**LangMem** adds durable, searchable, agent-managed memory on LangGraph's `BaseStore`:

1. **Hot path** — `manage_memory` / `search_memory` tools during conversation
2. **Background** — extract and consolidate facts after sessions
3. **Namespaces** — `("shopco", "{langgraph_user_id}")` for cross-thread recall
4. **Types** — semantic facts, episodic events, procedural behavior rules

ShopCo Support demonstrates preference persistence across new threads while keeping thread scratch in the checkpointer. Related topics: alternative LTM backends (e.g. Mem0) and **memory governance** for conflicts, decay, and audit trails.